## Проект "Викишоп" с BERT

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75.

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели.
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.
pip install pandas


In [4]:
pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Подготовка

In [1]:
# импортируем все необходимые библиотеки
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
from tqdm import notebook

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving toxic_comments.csv to toxic_comments.csv


In [ ]:
# прочитаем файл с комментариями
toxic_comments = pd.read_csv('toxic_comments.csv', index_col=0)

In [ ]:
# выведем первые 5 строк датафрейма
toxic_comments.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [ ]:
toxic_comments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 159292 entries, 0 to 159450
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


In [ ]:
toxic_comments.duplicated().sum()

np.int64(0)

## Baseline: TF-IDF + LogisticRegression



In [ ]:
# baseline: TF-IDF + LogisticRegression
text_col = "text"
target_col = "toxic"

if text_col not in toxic_comments.columns or target_col not in toxic_comments.columns:
    raise ValueError(f"Ожидались столбцы '{text_col}' и '{target_col}', но получены: {list(toxic_comments.columns)}")

X = toxic_comments[text_col].astype(str)
y = toxic_comments[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

vectorizer = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=100000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",
)
baseline_model.fit(X_train_tfidf, y_train)

y_pred = baseline_model.predict(X_test_tfidf)

f1 = f1_score(y_test, y_pred)
print(f"F1 (test): {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

В данных нет пропусков и дубликатов, приступим к предобработке текстов:

In [ ]:
# загрузка предобученной модели RuBERT и токенизатора
model_name = 'bert-base-cased'
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
# токенизируем текст c max_len = 512 (допустимое количество токенов в BertModel)
encoded = tokenizer(
    toxic_comments["text"].tolist(),
    add_special_tokens=True,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="np",
)

In [ ]:
padded = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

In [ ]:
batch_size = 256
embeddings = []
num_batches = (padded.shape[0] + batch_size - 1) // batch_size
# округляем вверх, так как число записей не делится нацело на 256

for i in notebook.tqdm(range(num_batches)):
    start_idx = batch_size * i
    end_idx = min(batch_size * (i + 1), padded.shape[0])

    batch = torch.LongTensor(padded[start_idx:end_idx])
    attention_mask_batch = torch.LongTensor(attention_mask[start_idx:end_idx])

    with torch.no_grad():
        batch_embeddings = model(batch, attention_mask=attention_mask_batch)

    embeddings.append(batch_embeddings[0][:,0,:].numpy())

  0%|          | 0/623 [00:00<?, ?it/s]